# Evaluate
Loads a saved checkpoint and computes redshift MAE, reconstruction MSE, and outlier rate.

In [ ]:
import os, sys
import torch
import torch.nn.functional as F
import numpy as np
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
import config
%run data.ipynb
%run model.ipynb

In [ ]:
def evaluate(model, loader, device):
    """Compute evaluation metrics on a held-out dataset split.

    Args:
        model (SpectralTransformer): Trained model in eval mode.
        loader (DataLoader): Data loader for the evaluation split.
        device (torch.device): Target compute device.

    Returns:
        dict: {
            'redshift_mae': mean absolute error on redshift predictions,
            'recon_mse': mean squared error on masked patch reconstruction,
            'outlier_rate': fraction of samples with |delta_z| > 0.1
        }
    """
    model.eval()
    all_z_pred, all_z_true, all_recon_mse = [], [], []

    with torch.no_grad():
        for flux, redshift in loader:
            flux = flux.to(device)
            redshift = redshift.unsqueeze(1).to(device)
            B = flux.shape[0]

            mask = random_mask(B, config.N_PATCHES).to(device)

            patches = flux.reshape(B, config.N_PATCHES, config.PATCH_SIZE)
            n_masked = mask.sum(dim=1)[0].item()
            target_patches = patches[mask].reshape(B, n_masked, config.PATCH_SIZE)

            recon_pred, z_pred = model(flux, mask)

            all_recon_mse.append(F.mse_loss(recon_pred, target_patches).item())
            all_z_pred.append(z_pred.cpu())
            all_z_true.append(redshift.cpu())

    z_pred_np = torch.cat(all_z_pred).numpy().flatten()
    z_true_np = torch.cat(all_z_true).numpy().flatten()
    delta_z = np.abs(z_pred_np - z_true_np)

    return {
        'redshift_mae': float(np.mean(delta_z)),
        'recon_mse':    float(np.mean(all_recon_mse)),
        'outlier_rate': float(np.mean(delta_z > 0.1)),
    }

In [ ]:
CHECKPOINT_PATH = os.path.join('..', 'checkpoints', 'best.pt')

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

model = SpectralTransformer().to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

_, val_loader = get_dataloaders(batch_size=256, n_train=1, n_val=2000)
metrics = evaluate(model, val_loader, device)

print(f"Redshift MAE:            {metrics['redshift_mae']:.4f}")
print(f"Reconstruction MSE:      {metrics['recon_mse']:.4f}")
print(f"Outlier rate (|Δz|>0.1): {metrics['outlier_rate']:.3f}")